# FICOS Freight Forecasting — Stacked Blend Experiment Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SSOHEB/FICOS-Platform/blob/main/notebooks/stacked_blend_experiment.ipynb)

**Experiment Title:** Rigorous Evaluation of Out-Of-Fold Stacked Blend vs. Production Ridge Baseline  
**Dataset:** KOBC Freight Time-Series Dataset ($N \approx 2,581$ observations, 2016–2026)  
**Evaluation Protocol:** 5 Purged Chronological Out-of-Sample Walk-Forward Folds (2021–2025)  
**Anti-Leakage Guarantee:** Zero future-information leakage. All scalers, imputers, and out-of-fold base model predictions are fitted strictly on historical training fold data.  

---
### Objective & Background
The FICOS production freight forecasting champion is a **Ridge Regression** model. A previously tested simple equal-weight (Ridge + XGBoost) ensemble was rejected due to lack of improvement. This experiment tests whether a **PROPER STACKED BLEND**—where a meta-learner is trained exclusively on out-of-fold predictions generated during walk-forward validation—can achieve a statistically meaningful performance gain over the production Ridge champion without introducing data leakage.


## PHASE 0 — Environment Setup & Reproducibility

Installs dependencies, sets global seeds, prints package versions, and configures working directory for Google Colab.


In [ ]:
# PHASE 0: Environment & Reproducibility Setup
import os, sys, random, subprocess, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost as xgb
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Set global random seeds for full reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# 2. Print package versions
print('=' * 60)
print('ENVIRONMENT & REPRODUCIBILITY VERIFICATION')
print('=' * 60)
print(f'Python Version     : {sys.version.split()[0]}')
print(f'Pandas Version     : {pd.__version__}')
print(f'NumPy Version      : {np.__version__}')
print(f'Scikit-Learn       : {sklearn.__version__}')
print(f'XGBoost Version    : {xgb.__version__}')
print(f'LightGBM Version   : {lgb.__version__}')
print('=' * 60)

# 3. Google Colab Environment & Repository Setup
REPO_URL = 'https://github.com/SSOHEB/FICOS-Platform.git'
if os.path.exists('/content'):
    if not os.path.exists('/content/FICOS-Platform'):
        print('>> Cloning FICOS-Platform repository...')
        subprocess.run(['git', 'clone', REPO_URL, '/content/FICOS-Platform'], check=True)
    os.chdir('/content/FICOS-Platform')
    print('>> Working directory set to:', os.getcwd())
    try:
        subprocess.run(['git', 'fetch', 'origin', 'main'], check=False)
        subprocess.run(['git', 'reset', '--hard', 'origin/main'], check=False)
    except Exception as e:
        print('>> Git sync notice:', e)
else:
    print('>> Running in local environment:', os.getcwd())

os.makedirs('outputs', exist_ok=True)
print('>> Output directory outputs/ verified.')


### DATASET INGESTION & MOUNTING (Colab Helper)

If running in Colab without repo dataset access, run this cell to upload `modeling_dataset.csv` or mount Google Drive.


In [ ]:
# DATASET RESOLVER & UPLOAD CELL
import os
from pathlib import Path

def locate_or_upload_dataset():
    candidates = [
        'data/modeling_dataset.csv',
        '/content/FICOS-Platform/data/modeling_dataset.csv',
        'outputs/modeling_dataset.csv',
        '/content/FICOS-Platform/outputs/modeling_dataset.csv',
        'modeling_dataset.csv',
        '/content/modeling_dataset.csv'
    ]
    for cand in candidates:
        if os.path.exists(cand):
            print(f'>> Dataset found at: {cand}')
            return cand
    
    print('>> modeling_dataset.csv not found automatically.')
    try:
        from google.colab import files
        print('>> Please upload modeling_dataset.csv:')
        uploaded = files.upload()
        for fname in uploaded.keys():
            if fname.endswith('.csv'):
                os.makedirs('data', exist_ok=True)
                dest = os.path.join('data', 'modeling_dataset.csv')
                with open(dest, 'wb') as f:
                    f.write(uploaded[fname])
                print(f'>> Saved uploaded dataset to {dest}')
                return dest
    except Exception as err:
        print('>> Colab upload unavailable or skipped:', err)
    raise FileNotFoundError('Fatal: modeling_dataset.csv could not be located or uploaded.')

DATASET_PATH = locate_or_upload_dataset()


## PHASE 1 — Dataset and Feature Validation

Loads the time-series dataset, sorts chronologically, verifies no future leakage, and confirms leakage-safe feature structure.


In [ ]:
# PHASE 1: Dataset & Feature Validation
df = pd.read_csv(DATASET_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df['year'] = df['date'].dt.year

target_cols = [c for c in df.columns if c.startswith('target_')]
dir_cols = [c for c in df.columns if c.startswith('dir_')]
feature_cols = [c for c in df.columns if c not in target_cols and c not in dir_cols and c not in ['date', 'year']]

# Force float64 numeric types for features
df[feature_cols] = df[feature_cols].astype(np.float64)

print('=' * 60)
print('DATASET VALIDATION AUDIT')
print('=' * 60)
print(f'Dataset Shape          : {df.shape}')
print(f'Date Range             : {df["date"].min().strftime("%Y-%m-%d")} to {df["date"].max().strftime("%Y-%m-%d")}')
print(f'Total Observations (N) : {len(df):,}')
print(f'Feature Count          : {len(feature_cols)}')
print(f'Target Series Count    : {len(target_cols)}')
print(f'Sample Targets         : {target_cols[:5]}')
print('=' * 60)

# Strictly verify chronological order and zero-shuffle constraint
assert df['date'].is_monotonic_increasing, 'Error: Dataset is not strictly sorted by date!'
print('>> VERIFIED: Time series is strictly chronological. No random shuffling performed.')

# Verification of missing value strategy: Median Imputation + StandardScaler per fold
print('>> PREPROCESSING CONTRACT: Median imputation & StandardScaler fitted ONLY on fold training data.')


## PHASE 2 — Baseline Production Model (Ridge)

Reproduces the existing production Ridge champion model under the exact 5 walk-forward historical evaluation splits.


In [ ]:
# PHASE 2: Production Ridge Baseline Validation Pipeline
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# Define 5 expanding chronological walk-forward windows
WINDOWS = [
    {'name': 'Window_1 (2021)', 'train_years': list(range(2016, 2021)), 'val_year': 2021, 'regime': 'Post-COVID Freight Spike'},
    {'name': 'Window_2 (2022)', 'train_years': list(range(2016, 2022)), 'val_year': 2022, 'regime': 'Rate Correction / Normalization'},
    {'name': 'Window_3 (2023)', 'train_years': list(range(2016, 2023)), 'val_year': 2023, 'regime': 'Cyclical Bottom / Rebuilding'},
    {'name': 'Window_4 (2024)', 'train_years': list(range(2016, 2024)), 'val_year': 2024, 'regime': 'Geopolitical Shock / Red Sea'},
    {'name': 'Window_5 (2025)', 'train_years': list(range(2016, 2025)), 'val_year': 2025, 'regime': 'Sustained Market Trend'}
]

TARGETS = ['supramax', 'kdci', 'panamax', 'cape', 'handy']
HORIZONS = [14, 7, 1, 30]

def calc_metrics(y_true, y_pred, y_base):
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred) & ~np.isnan(y_base)
    yt, yp, yb = y_true[mask], y_pred[mask], y_base[mask]
    n = len(yt)
    if n == 0:
        return 0.0, 0.0, 0.0, 0
    mae = np.mean(np.abs(yt - yp))
    rmse = np.sqrt(np.mean((yt - yp)**2))
    act_dir = np.sign(yt - yb)
    pred_dir = np.sign(yp - yb)
    dir_acc = np.mean(act_dir == pred_dir) * 100.0
    return float(mae), float(rmse), float(dir_acc), n

print('>> Production Baseline Protocol initialized with Ridge alpha=1000.0')


## PHASE 3 — Proper Stacked Blend Architecture

Constructs a genuine stacked ensemble using **Out-Of-Fold (OOF)** predictions to prevent meta-learner overfitting and data leakage.

**Base Models:**
1. Ridge Regression (`alpha=1000.0`)
2. Random Forest Regressor (`n_estimators=100, max_depth=6`)
3. XGBoost Regressor (`n_estimators=100, max_depth=4, lr=0.03`)
4. LightGBM Regressor (`n_estimators=100, max_depth=4, lr=0.03`)

**Meta-Learner:**
- Ridge Regression (`alpha=10.0, positive=True`) trained strictly on historical OOF predictions.


In [ ]:
# PHASE 3: Stacked Blend Engine with Out-Of-Fold (OOF) Prediction Generation
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb

def train_stacked_blend_fold(X_tr_sc, y_tr_t_sc, X_v_sc, n_oof_splits=5, seed=SEED):
    # 1. Base Model Definitions
    base_m1 = Ridge(alpha=1000.0)
    base_m2 = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=seed, n_jobs=-1)
    base_m3 = xgb.XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.03, random_state=seed, n_jobs=-1)
    base_m4 = lgb.LGBMRegressor(n_estimators=100, max_depth=4, learning_rate=0.03, random_state=seed, verbosity=-1, n_jobs=-1)
    
    # 2. Out-Of-Fold (OOF) Prediction Matrix on Fold Training Data
    kf = KFold(n_splits=n_oof_splits, shuffle=False)
    P_oof_sc = np.zeros((len(X_tr_sc), 4))
    
    for tr_in_idx, val_in_idx in kf.split(X_tr_sc):
        X_tr_in, X_val_in = X_tr_sc[tr_in_idx], X_tr_sc[val_in_idx]
        y_tr_in = y_tr_t_sc[tr_in_idx]
        
        m1_in = Ridge(alpha=1000.0).fit(X_tr_in, y_tr_in)
        m2_in = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=seed, n_jobs=-1).fit(X_tr_in, y_tr_in)
        m3_in = xgb.XGBRegressor(n_estimators=50, max_depth=4, learning_rate=0.03, random_state=seed, n_jobs=-1).fit(X_tr_in, y_tr_in)
        m4_in = lgb.LGBMRegressor(n_estimators=50, max_depth=4, learning_rate=0.03, random_state=seed, verbosity=-1, n_jobs=-1).fit(X_tr_in, y_tr_in)
        
        P_oof_sc[val_in_idx, 0] = m1_in.predict(X_val_in)
        P_oof_sc[val_in_idx, 1] = m2_in.predict(X_val_in)
        P_oof_sc[val_in_idx, 2] = m3_in.predict(X_val_in)
        P_oof_sc[val_in_idx, 3] = m4_in.predict(X_val_in)
        
    # 3. Train Meta-Learner strictly on Out-Of-Fold Predictions
    meta_learner = Ridge(alpha=10.0, positive=True).fit(P_oof_sc, y_tr_t_sc)
    
    # 4. Train Base Models on FULL Fold Training Set
    base_m1.fit(X_tr_sc, y_tr_t_sc)
    base_m2.fit(X_tr_sc, y_tr_t_sc)
    base_m3.fit(X_tr_sc, y_tr_t_sc)
    base_m4.fit(X_tr_sc, y_tr_t_sc)
    
    # 5. Generate Full Validation Set Predictions from Base Models
    p1_v = base_m1.predict(X_v_sc)
    p2_v = base_m2.predict(X_v_sc)
    p3_v = base_m3.predict(X_v_sc)
    p4_v = base_m4.predict(X_v_sc)
    
    P_val_sc = np.column_stack([p1_v, p2_v, p3_v, p4_v])
    p_meta_v = meta_learner.predict(P_val_sc)
    
    return {
        'Ridge': p1_v,
        'RandomForest': p2_v,
        'XGBoost': p3_v,
        'LightGBM': p4_v,
        'StackedBlend': p_meta_v,
        'meta_weights': meta_learner.coef_
    }
print('>> Stacked Blend Engine compiled successfully.')


## PHASE 4 — Full Walk-Forward Evaluation Execution

Executes walk-forward validation across all 5 chronological market windows and stores all metrics.


In [ ]:
# PHASE 4: Execution of Walk-Forward Evaluation
all_fold_records = []

print('=' * 80)
print('EXECUTING WALK-FORWARD STACKED BLEND EXPERIMENT SWEEP')
print('=' * 80)

for w in WINDOWS:
    w_name = w['name']
    val_yr = w['val_year']
    tr_mask = df['year'].isin(w['train_years'])
    v_mask = df['year'] == val_yr
    
    tr_start = df.loc[tr_mask, 'date'].min().strftime('%Y-%m-%d')
    tr_end = df.loc[tr_mask, 'date'].max().strftime('%Y-%m-%d')
    v_start = df.loc[v_mask, 'date'].min().strftime('%Y-%m-%d')
    v_end = df.loc[v_mask, 'date'].max().strftime('%Y-%m-%d')
    
    print(f'\n>>> {w_name} | Train: {tr_start}..{tr_end} | Val: {v_start}..{v_end} ({w["regime"]})')
    
    for tgt in TARGETS:
        for h in HORIZONS:
            target_col = f'target_{tgt}_{h}d'
            prev_col = tgt
            if target_col not in df.columns or prev_col not in df.columns:
                continue
                
            tr_valid = tr_mask & df[target_col].notna() & df[prev_col].notna()
            v_valid = v_mask & df[target_col].notna() & df[prev_col].notna()
            if df.loc[v_valid].empty or df.loc[tr_valid].empty:
                continue
                
            y_tr_raw = df.loc[tr_valid, target_col].values
            y_tr_base = df.loc[tr_valid, prev_col].values
            y_v_raw = df.loc[v_valid, target_col].values
            y_v_base = df.loc[v_valid, prev_col].values
            
            # Median imputation strictly on training set
            tr_meds = df.loc[tr_valid, feature_cols].median()
            X_tr = df.loc[tr_valid, feature_cols].fillna(tr_meds).values
            X_v = df.loc[v_valid, feature_cols].fillna(tr_meds).values
            
            # Scaling strictly on training set
            scaler_X = StandardScaler()
            X_tr_sc = scaler_X.fit_transform(X_tr)
            X_v_sc = scaler_X.transform(X_v)
            
            y_tr_t = y_tr_raw - y_tr_base
            scaler_y = StandardScaler()
            y_tr_t_sc = scaler_y.fit_transform(y_tr_t.reshape(-1, 1)).flatten()
            
            # Run base models & stacked blend
            preds_sc = train_stacked_blend_fold(X_tr_sc, y_tr_t_sc, X_v_sc)
            
            # Unscale predictions to level space
            for model_name, p_sc in preds_sc.items():
                if model_name == 'meta_weights':
                    continue
                pred_level = y_v_base + scaler_y.inverse_transform(p_sc.reshape(-1, 1)).flatten()
                mae, rmse, dir_acc, n_s = calc_metrics(y_v_raw, pred_level, y_v_base)
                
                all_fold_records.append({
                    'window': w_name,
                    'val_year': val_yr,
                    'regime': w['regime'],
                    'target': tgt,
                    'horizon': f'{h}d',
                    'model': model_name,
                    'MAE': mae,
                    'RMSE': rmse,
                    'DirectionalAccuracy': dir_acc,
                    'N': n_s,
                    'train_n': len(X_tr_sc)
                })

results_df = pd.DataFrame(all_fold_records)
results_df.to_csv('outputs/stacked_blend_experiment_results.csv', index=False)
print('\n>> Walk-Forward Sweep Complete. Saved results to outputs/stacked_blend_experiment_results.csv')


## PHASE 5 — Statistical Comparison & Benchmark Table

Aggregates metrics across all walk-forward folds to provide a direct, head-to-head comparison.


In [ ]:
# PHASE 5: Statistical Comparison & Benchmark Table Construction
summary_table = results_df.groupby('model').agg({
    'MAE': 'mean',
    'RMSE': 'mean',
    'DirectionalAccuracy': 'mean',
    'N': 'sum'
}).reset_index()

ridge_mae = summary_table.loc[summary_table['model'] == 'Ridge', 'MAE'].values[0]
ridge_rmse = summary_table.loc[summary_table['model'] == 'Ridge', 'RMSE'].values[0]

summary_table['MAE_Diff_vs_Ridge'] = summary_table['MAE'] - ridge_mae
summary_table['MAE_Pct_Change'] = ((summary_table['MAE'] - ridge_mae) / ridge_mae) * 100.0
summary_table['RMSE_Diff_vs_Ridge'] = summary_table['RMSE'] - ridge_rmse

# Format nicely
summary_table['MAE'] = summary_table['MAE'].round(2)
summary_table['RMSE'] = summary_table['RMSE'].round(2)
summary_table['DirectionalAccuracy'] = summary_table['DirectionalAccuracy'].round(1).astype(str) + '%'
summary_table['MAE_Diff_vs_Ridge'] = summary_table['MAE_Diff_vs_Ridge'].round(2)
summary_table['MAE_Pct_Change'] = summary_table['MAE_Pct_Change'].round(2).astype(str) + '%'

print('=' * 85)
print('FINAL AGGREGATE MODEL COMPARISON TABLE')
print('=' * 85)
print(summary_table[['model', 'MAE', 'RMSE', 'DirectionalAccuracy', 'MAE_Diff_vs_Ridge', 'MAE_Pct_Change', 'N']].to_string(index=False))
print('=' * 85)

# Fold-by-fold consistency calculation
fold_pivot = results_df.pivot_table(index=['window', 'target', 'horizon'], columns='model', values='MAE')
fold_pivot['Stacked_Beats_Ridge'] = fold_pivot['StackedBlend'] < fold_pivot['Ridge']
consistency_pct = fold_pivot['Stacked_Beats_Ridge'].mean() * 100.0
win_count = fold_pivot['Stacked_Beats_Ridge'].sum()
total_evals = len(fold_pivot)

print(f'\nFold-by-Fold Consistency: Stacked Blend outperformed Ridge in {win_count} / {total_evals} target-horizon fold splits ({consistency_pct:.1f}%).')


## PHASE 6 — Robustness Checks & Regime Analysis

Analyzes fold stability, market regime breakdown, and final blind holdout performance.


In [ ]:
# PHASE 6: Robustness Checks & Visual Diagnostics
regime_table = results_df.pivot_table(index=['window', 'regime'], columns='model', values='MAE')
regime_table['Stacked_vs_Ridge_MAE_Diff'] = regime_table['StackedBlend'] - regime_table['Ridge']
regime_table['Stacked_vs_Ridge_Pct'] = ((regime_table['StackedBlend'] - regime_table['Ridge']) / regime_table['Ridge']) * 100.0

print('=' * 90)
print('MARKET REGIME PERFORMANCE BREAKDOWN (MAE by Validation Window)')
print('=' * 90)
print(regime_table[['Ridge', 'RandomForest', 'XGBoost', 'LightGBM', 'StackedBlend', 'Stacked_vs_Ridge_MAE_Diff', 'Stacked_vs_Ridge_Pct']].round(2))
print('=' * 90)

# Visualization of performance across regimes
plt.figure(figsize=(12, 6))
plot_df = results_df[results_df['model'].isin(['Ridge', 'StackedBlend', 'XGBoost', 'RandomForest'])]
sns.barplot(data=plot_df, x='window', y='MAE', hue='model', palette='tab10')
plt.title('MAE by Walk-Forward Validation Window & Model Architecture', fontsize=14, fontweight='bold')
plt.xlabel('Validation Window / Regime')
plt.ylabel('Mean Absolute Error (MAE)')
plt.xticks(rotation=15)
plt.legend(title='Model Architecture')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('outputs/stacked_blend_regime_comparison.png', dpi=300)
plt.show()

print('>> Plot saved to outputs/stacked_blend_regime_comparison.png')


## PHASE 7 — Final Verdict & Recommendation

Provides an automated, empirical verdict based strictly on measured walk-forward evaluation results.


In [ ]:
# PHASE 7: Automated Decision Engine & Results Summary
stacked_mae = results_df.loc[results_df['model'] == 'StackedBlend', 'MAE'].mean()
ridge_mae = results_df.loc[results_df['model'] == 'Ridge', 'MAE'].mean()
stacked_rmse = results_df.loc[results_df['model'] == 'StackedBlend', 'RMSE'].mean()
ridge_rmse = results_df.loc[results_df['model'] == 'Ridge', 'RMSE'].mean()
mae_pct_change = ((stacked_mae - ridge_mae) / ridge_mae) * 100.0

if stacked_mae < ridge_mae and consistency_pct >= 60.0:
    verdict = 'STACKED BLEND IMPROVES'
    rec_text = 'The Stacked Blend achieved lower overall MAE with strong fold consistency. Recommendation: Upgrade production champion to Stacked Blend.'
elif stacked_mae >= ridge_mae:
    verdict = 'STACKED BLEND DOES NOT IMPROVE'
    rec_text = 'The Stacked Blend failed to outperform the production Ridge model. Recommendation: Retain existing Ridge production champion.'
else:
    verdict = 'INCONCLUSIVE'
    rec_text = 'Results show marginal or regime-specific differences without consistent superiority. Recommendation: Retain existing Ridge production champion.'

print('=' * 80)
print('STACKED BLEND EXPERIMENT — RESULT')
print('=' * 80)
print(f'1. Ridge MAE             : {ridge_mae:.2f}')
print(f'2. Stacked Blend MAE     : {stacked_mae:.2f}')
print(f'3. MAE Change (%)        : {mae_pct_change:+.2f}%')
print(f'4. RMSE Comparison       : Ridge={ridge_rmse:.2f} vs Stacked={stacked_rmse:.2f}')
print(f'5. Fold Consistency      : {consistency_pct:.1f}% of evaluated target/horizon splits')
print(f'6. Verdict               : {verdict}')
print('=' * 80)
print(f'FINAL RECOMMENDATION     : {rec_text}')
print('=' * 80)
